In [1]:
import pandas as pd
from collections import deque
pd.options.display.float_format = "{:,.2f}".format


# Load Data

In [2]:
df_production = pd.read_csv('../data/production.csv', delimiter=';')
df_exports_companies = pd.read_csv('../data/branches_companies.csv', delimiter=';').rename(columns={'logistics_hub.trase_id': 'source_id'})
df_exports = pd.read_csv('../data/branches.csv', delimiter=';')
df_solver = pd.read_csv('../data/2023/relatorio_auditoria_solver.csv', delimiter=';').query('Categoria in ("REAL_LOGISTICA")')

# Data Preparation

In [3]:
df_solver = df_solver.rename(columns={'Destino/Info': 'Destination_ID'})

mask_1 = df_exports_companies['branch'].str.startswith('1')
df_exports_companies.loc[mask_1, 'source_id'] = df_exports_companies.loc[mask_1, 'source_id'].str.replace('BR', 'PRODUCTION-BR')

mask_2 = df_exports_companies['branch'].str.startswith('2')
df_exports_companies.loc[mask_2, 'source_id'] = df_exports_companies.loc[mask_2, 'source_id'].str.replace('BR', 'HUB-BR')

mask_3 = df_exports_companies['branch'].str.startswith('3')
df_exports_companies.loc[mask_3, 'source_id'] = df_exports_companies.loc[mask_3, 'source_id'].str.replace('BR', 'PROCESSING-BR')

df_exports_companies['port_municipality_trase_id'] = df_exports_companies['port_municipality_trase_id'].apply(lambda trase_id: str(trase_id).replace('BR', 'PORT-BR'))

df_exports_companies = df_exports_companies[['exporter_group', 'source_id', 'port_municipality_trase_id', 'volume', 'branch','product_type']]

df_exports_companies = df_exports_companies.groupby(
    by=['exporter_group', 'source_id', 'port_municipality_trase_id', 'branch','product_type'],
    as_index=False
)['volume'].sum()

In [4]:
df_exports[['product_type','volume']].groupby('product_type').sum()

,volume
product_type,
SOYBEAN,"94,396,387.99"
SOYBEAN_CAKE,"20,569,076.18"
SOYBEAN_OIL,"2,082,346.46"


In [30]:
df_exports_companies[['volume']].sum()

volume   116,366,730.02
dtype: float64

In [31]:
df_exports[['volume']].sum()

volume   117,047,810.62
dtype: float64

In [32]:
df_production[['volume']].sum()

volume   152,144,238.00
dtype: float64

In [33]:
df_solver[['Categoria','Volume']].groupby('Categoria').sum()

,Volume
Categoria,
REAL_LOGISTICA,"346,040,439.63"


In [34]:
df_solver[df_solver['Destination_ID'].str.startswith('PORT-')][['Volume']].sum()

Volume   118,574,204.86
dtype: float64

In [35]:

import numpy as np
from collections import defaultdict
import sys



# Aumentar o limite de recursão para redes logísticas muito profundas
sys.setrecursionlimit(5000)


df_exports_companies = df_exports_companies.query('branch not in ("UNPROCESSED", "UNKOWN")')


def proportional_merge(df_left, df_right, left_key, right_key, vol_col_left, vol_col_right):
    
    # 1. Garantir que as chaves sejam listas
    if isinstance(left_key, str): left_key = [left_key]
    if isinstance(right_key, str): right_key = [right_key]
        
    df_l = df_left.copy()
    df_r = df_right.copy()
    
    # 2. LIMPEZA DE DADOS (Blindada contra formatações mistas)
    def clean_numbers(col_data):
        if col_data.dtype == 'object':
            # Remove espaços em branco
            col_data = col_data.astype(str).str.strip()
            # Se tiver padrão BR (ponto no milhar e vírgula no decimal), ajusta:
            # Ex: 1.500,50 -> 1500.50
            if col_data.str.contains(r'\d+\.\d{3},\d+', regex=True).any():
                col_data = col_data.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
            else:
                # Se for padrão Americano (vírgula no milhar), apenas remove a vírgula
                col_data = col_data.str.replace(',', '', regex=False)
        
        return pd.to_numeric(col_data, errors='coerce').fillna(0)

    df_l[vol_col_left] = clean_numbers(df_l[vol_col_left])
    df_r[vol_col_right] = clean_numbers(df_r[vol_col_right])

    # 3. PREPARAÇÃO PARA EVITAR DUPLICAÇÃO DE VOLUME (CROSS-JOIN)
    df_l['_row_id'] = range(len(df_l))
    
    # Calcula a capacidade total daquela rota e o peso de cada linha (ex: caminhão vs trem)
    left_totals = df_l.groupby(left_key)[vol_col_left].transform('sum')
    df_l['weight'] = np.where(left_totals > 0, df_l[vol_col_left] / left_totals, 0)

    # PASSO 1: O Match
    df_matched = pd.merge(df_l, df_r, left_on=left_key, right_on=right_key, how='inner') 

    # Fatiar a carga do exportador proporcionalmente à capacidade física da linha
    df_matched['allocated_vol_right'] = df_matched[vol_col_right] * df_matched['weight']
    df_matched[vol_col_right] = df_matched['allocated_vol_right']

    df_matched['proportion'] = np.where(
        df_matched[vol_col_left] == 0, 
        0, 
        (df_matched[vol_col_right] / df_matched[vol_col_left])
    ).round(4)
    df_matched['match_indicator'] = True

    # PASSO 2: Descobrir o Saldo LINHA A LINHA
    used_capacity = df_matched.groupby('_row_id')[vol_col_right].sum()
    df_l['used_vol'] = df_l['_row_id'].map(used_capacity).fillna(0)
    
    # Saldo = Volume original da aresta - O que os exportadores consumiram
    df_l['leftover_vol'] = df_l[vol_col_left] - df_l['used_vol']

    # PASSO 3: Criar as linhas do saldo remanescente (No Match)
    # Tolerância de > 0.001 para evitar criar linhas de "sujeira" por imprecisão de float
    df_leftovers = df_l[df_l['leftover_vol'] > 0.001].copy() 
    
    if not df_leftovers.empty:
        df_leftovers[vol_col_right] = df_leftovers['leftover_vol']
        
        df_leftovers['proportion'] = np.where(
            df_leftovers[vol_col_left] == 0,
            0,
            (df_leftovers[vol_col_right] / df_leftovers[vol_col_left])
        ).round(4)
        
        df_leftovers['match_indicator'] = False
        
        for l_k, r_k in zip(left_key, right_key):
            df_leftovers[r_k] = df_leftovers[l_k]
        
        extra_cols = [col for col in df_r.columns if col not in right_key and col != vol_col_right]
        for col in extra_cols:
            df_leftovers[col] = 'No Match'

    # PASSO 4: Empilhamento Final
    cols_to_drop = ['_row_id', 'weight', 'allocated_vol_right', 'used_vol', 'leftover_vol']
    df_matched = df_matched.drop(columns=[c for c in cols_to_drop if c in df_matched.columns])
    df_leftovers = df_leftovers.drop(columns=[c for c in cols_to_drop if c in df_leftovers.columns], errors='ignore')
            
    final_cols = list(df_matched.columns) 
    
    if not df_leftovers.empty:
        df_final = pd.concat([df_matched, df_leftovers[final_cols]], ignore_index=True)
    else:
        df_final = df_matched.copy()

    sort_cols = left_key + ['match_indicator']
    sort_dirs = [True] * len(left_key) + [False]
    df_final = df_final.sort_values(by=sort_cols, ascending=sort_dirs).reset_index(drop=True)

    return df_final


def mass_balance_back_propagation(df):
    # 1. CONSTRUIR O GRAFO FÍSICO LIMPO (Sem as duplicações do Merge)
    # Pegamos apenas as conexões únicas e seus volumes físicos reais ('Volume' maiúsculo)
    df_edges = df[['Node_ID', 'Destination_ID', 'Volume']].drop_duplicates()
    
    capacity = defaultdict(dict)
    for _, row in df_edges.iterrows():
        if pd.notna(row['Node_ID']) and pd.notna(row['Destination_ID']):
            capacity[row['Destination_ID']][row['Node_ID']] = row['Volume']
            
    # 2. DEFINIR O GATILHO (Demanda)
    # Vamos pegar TUDO que tem 'PORT' no destino, garantindo que o 100% do volume seja rastreado
    demands = df[df['Destination_ID'].astype(str).str.contains('PORT', na=False, case=False)].copy()
    
    completed_paths = []
    
    # 3. O MOTOR DE RASTREAMENTO RECURSIVO (Não perde 1 grama)
    def trace_back(current_node, demand_vol, path, exporter, product):
        if demand_vol <= 0: return
        
        preds = capacity.get(current_node, {})
        # Filtra apenas os predecessores que ainda têm saldo no grafo
        available_preds = {p: v for p, v in preds.items() if v > 0}
        total_avail = sum(available_preds.values())
        
        # Condição de Parada: Não tem mais de onde puxar (Origem)
        if total_avail == 0:
            completed_paths.append({
                'Path': path,
                'Allocated_Volume': demand_vol, # Salva o volume EXATO
                'Exporter': exporter,
                'Product': product
            })
            return
            
        # Distribui o volume estritamente proporcional ao saldo disponível
        for p, avail in available_preds.items():
            proportion = avail / total_avail
            pull_vol = demand_vol * proportion
            
            # Condição de Parada 2: Evitar Loop infinito (Ex: Silo A -> Silo B -> Silo A)
            if p in path:
                completed_paths.append({
                    'Path': [p] + path,
                    'Allocated_Volume': pull_vol,
                    'Exporter': exporter,
                    'Product': product
                })
                continue
            
            # Subtrai do Grafo Global
            capacity[current_node][p] -= pull_vol
            # Garante que não fique negativo por erro microscópico de float do Python
            if capacity[current_node][p] < 0: 
                capacity[current_node][p] = 0
            
            # Chama a si mesmo para dar o próximo passo para trás
            trace_back(p, pull_vol, [p] + path, exporter, product)

    # 4. EXECUTAR PARA CADA LINHA QUE CHEGA NO PORTO
    for _, row in demands.iterrows():
        port = row['Destination_ID']
        hub = row['Node_ID']
        # Usamos o 'volume' (minúsculo) pois ele tem a quebra exata do merge e do saldo
        vol_to_trace = row['volume'] 
        exporter = row.get('exporter_group', 'Unknown')
        product = row.get('Produto', 'Unknown')
        
        # Abate o volume da aresta inicial (Hub -> Porto)
        if hub in capacity.get(port, {}):
            capacity[port][hub] = max(0, capacity[port][hub] - vol_to_trace)
            
        # Inicia a jornada de volta
        trace_back(hub, vol_to_trace, [hub, port], exporter, product)

    # 5. CONVERSÃO PARA WIDE TABLE
    df_results = pd.DataFrame(completed_paths)
    if df_results.empty: 
        return df_results
    
    max_steps = df_results['Path'].apply(len).max()
    for i in range(max_steps):
        df_results[f'Step_{i+1}'] = df_results['Path'].apply(lambda x: x[i] if i < len(x) else None)
        
    df_results = df_results.drop(columns=['Path'])
    
    # Organiza as colunas: Steps primeiro, depois os dados
    cols = [c for c in df_results.columns if 'Step_' in c] + ['Exporter', 'Product', 'Allocated_Volume']
    return df_results[cols]




In [36]:
df_consolidated = proportional_merge(
    df_solver,
    df_exports_companies[['exporter_group', 'source_id', 'port_municipality_trase_id', 'volume', 'branch','product_type']],
    left_key=['Node_ID', 'Destination_ID', 'Produto'],
    right_key=['source_id', 'port_municipality_trase_id', 'product_type'],
    vol_col_left='Volume',
    vol_col_right='volume'
)

df_wide_paths = mass_balance_back_propagation(df_consolidated)


In [37]:
df_consolidated[
    (df_consolidated['Destination_ID'].str.startswith('PORT-')) &
    (~df_consolidated['source_id'].str.startswith('TRAIN-'))
][['volume']].sum()

volume   79,832,232.08
dtype: float64

In [38]:
df_consolidated[
    (df_consolidated['branch'].str.startswith('2'))
][['volume']].sum()

volume   28,192,071.51
dtype: float64

In [39]:
df_exports_companies[
    (df_exports_companies['branch'].str.startswith('2'))
][['volume']].sum()

volume   29,251,393.65
dtype: float64

In [40]:
A = df_consolidated[
    (df_consolidated['branch'].str.startswith('2'))
][['source_id', 'port_municipality_trase_id']]

B = df_exports_companies[
    (df_exports_companies['branch'].str.startswith('2'))
][['source_id', 'port_municipality_trase_id', 'volume']]

B.merge(
    A,
    how='outer',
    on=['source_id', 'port_municipality_trase_id'],
    indicator=True
).query('_merge == "left_only"').drop_duplicates()

,source_id,port_municipality_trase_id,volume,_merge
1,HUB-BR-1301902,PORT-BR-2111300,"66,740.00",left_only
21,HUB-BR-2100808,PORT-BR-4118204,"58,140.00",left_only
32,HUB-BR-2111300,PORT-BR-1302603,"165,000.00",left_only
46,HUB-BR-3205309,PORT-BR-1501303,"181,653.64",left_only
129,HUB-BR-4115200,PORT-BR-2111300,"2,000.00",left_only
169,HUB-BR-4208450,PORT-BR-2111300,"57,812.50",left_only
187,HUB-BR-5103403,PORT-BR-1302603,"184,160.00",left_only
188,HUB-BR-5103403,PORT-BR-2111300,"10,400.00",left_only
196,HUB-BR-5103502,PORT-BR-1302603,"265,916.00",left_only
201,HUB-BR-5107602,PORT-BR-1302603,"45,000.00",left_only


In [41]:
df_consolidated.to_csv('data/consolidated.csv', index=False, sep=';')

df_wide_paths.to_csv('data/consolidated_wide.csv', index=False, sep=';')

In [42]:
df_wide_paths.query('Exporter != "No Match" and Step_2.str.startswith("HUB-")')[['Allocated_Volume']].sum()

Allocated_Volume   28,279,416.50
dtype: float64

In [43]:
df_consolidated.query('Destination_ID.str.startswith("PORT-")')[['volume']].sum()

volume   120,729,593.39
dtype: float64

In [44]:
df_wide_paths[['Allocated_Volume']].sum()

Allocated_Volume   120,729,593.39
dtype: float64

In [45]:
df_wide_paths.query('Exporter != "No Match"')[['Allocated_Volume']].sum()

Allocated_Volume   42,855,689.79
dtype: float64

In [46]:
df_wide_paths.query('Exporter.str.startswith("No Match")')[['Allocated_Volume']].sum()

Allocated_Volume   77,873,903.60
dtype: float64

In [47]:
df_wide_paths[['Allocated_Volume']].sum()

Allocated_Volume   120,729,593.39
dtype: float64

In [48]:
df_wide_paths.query('Exporter.str.startswith("CARGILL")')[['Step_2']].drop_duplicates()

,Step_2
106,HUB-BR-1506807
345,HUB-BR-2101400
349,HUB-BR-2109007
432,HUB-BR-2903201
471,HUB-BR-3170206
488,HUB-BR-3528403
506,HUB-BR-3550308
12651,HUB-BR-4119905
12679,HUB-BR-4213609
14042,PROCESSING-BR-4119905


In [49]:
#Tipo;Origin;Destination;Product;Volume;Variavel_Solver
# 
df_consolidated_tool = df_consolidated[[
    'Categoria',
    'Node_ID',
    'Destination_ID',
    'exporter_group',
    'Produto',
    'volume',
    'Variavel_Solver'
]].rename(columns={
    'Categoria': 'Tipo',
    'exporter_group': 'Exporter',
    'Node_ID': 'Origin',
    'volume':'Volume',
    'Destination_ID': 'Destination',
    'Produto': 'Product'
})

In [50]:
df_consolidated_tool.to_csv('../tools/supply_shed_flows.csv', sep=';', index=False)

# python3 -m http.server 8000 